In [ ]:
import pandas as pd

path = "filtered.csv"
df = pd.read_csv(path)

df["ModelName"] = (
    df["Model"].astype(str)
    .str.extract(r"\[(.*?)\]\(", expand=False)
    .fillna(df["Model"].astype(str))
)

num_cols = [
    "Active Parameters (B)",
    "Total Parameters (B)",
    "Mean (Task)",
    "Retrieval",
    "Reranking",
    "Pair Classification",
]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

ranked = df[
    (df["Active Parameters (B)"] <= 0.3)
    & df["Total Parameters (B)"].notna()
    & df[["Mean (Task)", "Retrieval", "Reranking", "Pair Classification"]].notna().all(axis=1)
].copy()

ranked["SoftSort"] = (
    0.60 * ranked["Retrieval"]
    + 0.25 * ranked["Reranking"]
    + 0.15 * ranked["Pair Classification"]
)

ranked = ranked.sort_values(["SoftSort", "Retrieval"], ascending=False).reset_index(drop=True)

cols = [
    "ModelName",
    "Active Parameters (B)",
    "Total Parameters (B)",
    "Mean (Task)",
    "Retrieval",
    "Reranking",
    "Pair Classification",
    "SoftSort",
]

ranked[cols].head(20)

In [ ]:
remove_reason = {
    "jina-embeddings-v5-text-nano": "non-commercial license",
    "snowflake-arctic-embed-m-v2.0": "runtime instability",
    "BidirLM-270M-Embedding": "runtime instability",
    "bilingual-embedding-base": "English-French only",
    "bilingual-embedding-small": "English-French only",
    "granite-embedding-278m-multilingual": "superseded by 311m-r2",
    "granite-embedding-107m-multilingual": "superseded by 97m-r2",
    "F2LLM-v2-80M": "weaker smaller variant",
    "paraphrase-multilingual-mpnet-base-v2": "general paraphrase model",
    "paraphrase-multilingual-MiniLM-L12-v2": "general paraphrase model",
    "Arabic-all-nli-triplet-Matryoshka": "language/domain mismatch",
    "granite-embedding-125m-english": "English-only",
    "bge-base-en-v1.5": "English-only",
}

top = ranked.head(20).copy()
top["RemoveReason"] = top["ModelName"].map(remove_reason)

final = top[top["RemoveReason"].isna()].copy()
removed = top[top["RemoveReason"].notna()].copy()

final[cols]

In [ ]:
# removed[["ModelName", "RemoveReason", "Retrieval", "Reranking", "Pair Classification", "SoftSort"]]